# Introduction:

* Session = user, Aid = product

The competition's objective is to predict e-commerce clicks, cart additions, and orders using information from previous user session events. There are three types of interactions and 1671803 unique sessions overall (Clicks, Carts and Orders). Therefore, 5015409 predictions (1671803*3) would be made.

Details could be found in this page: https://www.kaggle.com/competitions/otto-recommender-system/overview/evaluation

# In short: 

The only product that needs to be predicted for the click prediction is the **next one** the user will click, for instances: aid1. However, we can submit a list of up to 20 products for prediction; **as long as aid1 is on the list, the score will be awarded.**

We could also submit a list of products (maximum 20) for prediction for cart and orders.  For example, the user adds **30** products into **cart**, and **order 3** products.

The maximum size of a prediction is 20, so **even though there are 30 products in the cart, the full score can still be obtained as long as every item on the prediction list appears in the cart**.

Since 3 products were ordered, I can still **receive a full score as long as all three appear on the prediction list.**




# Method:

I would predict the next aid(s) by only the **last 20 aids that the users have interacted with.**


In [ ]:
# Import Module
from collections import OrderedDict
import pandas as pd
import numpy as np
import warnings
# ignore all warnings
warnings. filterwarnings("ignore")
print("Imported")

## Load Data
df = pd.read_parquet('/kaggle/input/otto-full-optimized-memory-footprint/test.parquet')
print(df)

As there are cold users, users' interactions less than the max prediction 20, we would add the top 20 interacted items into the prediction list

In [ ]:
productcount = pd.DataFrame()
productcount["count"] = df['aid'].value_counts()
top20 = productcount["count"].tolist()[:20]
print(top20)
print(productcount)

In [ ]:
# # Get last 20 unique aids for each session

df = df.sort_values(["session", "type", "ts"]).groupby(["session"]).apply(
    lambda x: list(OrderedDict.fromkeys(x.tail(40).aid.tolist() + top20).keys())[:20])
print(df)
print("=="*32)

submission = pd.DataFrame()
for i in ['_clicks','_carts','_orders']:
    new_df = pd.DataFrame(df.add_suffix(str(i)), columns=["labels"]).reset_index()
    print(new_df)
    submission = pd.concat(
        [new_df, submission]
    )

submission = submission.rename(columns={'session': 'session_type'})
submission["labels"] = submission['labels'].apply(lambda x: " ".join(str(i) for i in x))
print(submission)
submission.to_csv("submission.csv", index=False)